In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType

spark = SparkSession.builder.appName("Pyspark_Final").getOrCreate()

print("Подключение создано")

schema = StructType([
    StructField("ip", StringType(), True),
    StructField("timestamp", DateType(), True),
    StructField("method", StringType(), True),
    StructField("url", StringType(), True),
    StructField("response_code", IntegerType(), True),
    StructField("response_size", IntegerType(), True)
])

df_logs = spark.read.schema(schema).option("header", "true").csv("/content/web_server_logs.csv")

print("Проверка датафрейма")
df_logs.show()
print("Датафрейм создан")



from pyspark.sql.functions import count, col, sum

# Группирует данные по IP и считает количество запросов для каждого IP, выводим 10 самых активных IP:
top_10 = df_logs.groupBy("ip").agg(count("url").alias("request_count")).orderBy(col("request_count").desc())
print("Top 10  active IP addresses:")
top_10.show(10)

# Группирует данные по HTTP-методу и считает количество запросов для каждого метода:
method_cnt_url = df_logs.groupBy("method").agg(count("url").alias("method_count"))
print("Request count by HTTP method:")
method_cnt_url.show()

# Фильтрует и считает количество запросов с кодом ответа 404:
count_404 = df_logs.filter(col("response_code") == 404).count()
print(f"\nNumber of 404 response codes: {count_404}\n")

# Группирует данные по дате и суммирует размер ответов, сортирует по дате:
size = df_logs.groupBy("timestamp").agg(sum("response_size").alias("total_response_size")).orderBy("timestamp")
print("Total response size by day:")
size.show()

TypeError: code() argument 13 must be str, not int